In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
path = "/home/ccm/models/Qwen2.5-7B-Instruct"

/home/ccm/anaconda3/envs/transformers/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = AutoModelForCausalLM.from_pretrained(path, device_map="cuda:3")
tokenizer = AutoTokenizer.from_pretrained(path)

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.72it/s]


In [3]:
sentence = "Hello!"

In [4]:
text = tokenizer.apply_chat_template(
    [
        {'role':'system', 'content':'You are a helpful assistant.'},
        {'role':'user', 'content': sentence}
    ]
    , 
    tokenize=False)
print(text)

text2 = tokenizer.apply_chat_template(
    [
        {'role':'system', 'content':'You are a helpful assistant.'},
        {'role':'user', 'content': "Hi!"}
    ]
    , 
    tokenize=False)
print(text2)


<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hello!<|im_end|>

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hi!<|im_end|>



In [7]:
model_input = tokenizer([text, text2], return_tensors='pt').to('cuda:3')
print(model_input)
print(model_input['input_ids'].shape)

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,   9707,      0, 151645,    198],
        [151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,  13048,      0, 151645,    198]],
       device='cuda:3'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:3')}
torch.Size([2, 18])


In [18]:
model_output = model(**model_input)
print(model_output)

CausalLMOutputWithPast(loss=None, logits=tensor([[[  4.3198,   6.1736,   6.7401,  ...,  -3.6969,  -3.6968,  -3.6970],
         [  3.1169,   1.8225,   7.6177,  ...,  -2.7360,  -2.7359,  -2.7359],
         [  3.4049,   5.3616,   7.3858,  ...,  -5.5598,  -5.5595,  -5.5598],
         ...,
         [ -9.6380,  -5.4570,  -3.3568,  ...,  -2.9716,  -2.9716,  -2.9721],
         [ -7.2810, -14.2610,  -6.4463,  ...,   4.4090,   4.4094,   4.4094],
         [-23.3222, -28.5762, -17.2127,  ...,  16.9050,  16.9050,  16.9054]],

        [[  4.3198,   6.1736,   6.7401,  ...,  -3.6969,  -3.6969,  -3.6970],
         [  3.1169,   1.8225,   7.6177,  ...,  -2.7360,  -2.7359,  -2.7359],
         [  3.4049,   5.3615,   7.3857,  ...,  -5.5598,  -5.5595,  -5.5598],
         ...,
         [ -8.5635,  -5.4190,  -4.0087,  ...,  -3.4089,  -3.4089,  -3.4095],
         [ -7.5296, -14.5670,  -6.8074,  ...,   4.3493,   4.3496,   4.3496],
         [-23.3956, -28.6367, -17.4611,  ...,  17.0211,  17.0210,  17.0214]]],
   

In [19]:
logits = model_output.logits
print(logits.shape)  # Shape like: [N, S, V] N - Batch Size, S - Sequence length, V - Vocabulary quantity


torch.Size([2, 18, 152064])


In [20]:
last_token_logits = logits[:, -1, :]  # Shape like: [N, V]
print(last_token_logits.shape)

torch.Size([2, 152064])


In [25]:
next_token_ids = torch.argmax(last_token_logits, dim=-1)
print(next_token_ids) 

tensor([151644, 151644])


In [26]:
next_token = tokenizer.batch_decode(next_token_ids)
print(next_token)

['<|im_start|>', '<|im_start|>']


In [27]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (rotary_emb):

In [8]:
model_output = model.generate(**model_input, max_new_tokens=50)
print(model_output)

tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,   9707,      0, 151645,    198,
         151644,  14990,      0,   2585,    646,    358,   7789,    498,   3351,
             30, 151645],
        [151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,  13048,      0, 151645,    198,
         151644,  14990,      0,   2585,    646,    358,   7789,    498,   3351,
             30, 151645]], device='cuda:3')


In [33]:
output_texts = tokenizer.batch_decode(model_output)
for text in output_texts:
    print(text)
    print()

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hello!<|im_end|>
<|im_start|> greetings!
Hello! How can I assist you today?<|im_end|>

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hi!<|im_end|>
<|im_start|>>Hello! How can I assist you today?<|im_end|><|endoftext|><|endoftext|>



In [38]:
input_ids_1 = model_input['input_ids'][0]
ai_message_start_idx = len(input_ids_1)
ai_message_token_ids = model_output[0][ai_message_start_idx:]
output_text_1 = tokenizer.decode(ai_message_token_ids, skip_special_tokens=True)
print(output_text_1)


 greetings!
Hello! How can I assist you today?


In [39]:
tokenizer.chat_template

'{%- if tools %}\n    {{- \'<|im_start|>system\\n\' }}\n    {%- if messages[0][\'role\'] == \'system\' %}\n        {{- messages[0][\'content\'] }}\n    {%- else %}\n        {{- \'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.\' }}\n    {%- endif %}\n    {{- "\\n\\n# Tools\\n\\nYou may call one or more functions to assist with the user query.\\n\\nYou are provided with function signatures within <tools></tools> XML tags:\\n<tools>" }}\n    {%- for tool in tools %}\n        {{- "\\n" }}\n        {{- tool | tojson }}\n    {%- endfor %}\n    {{- "\\n</tools>\\n\\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\\n<tool_call>\\n{\\"name\\": <function-name>, \\"arguments\\": <args-json-object>}\\n</tool_call><|im_end|>\\n" }}\n{%- else %}\n    {%- if messages[0][\'role\'] == \'system\' %}\n        {{- \'<|im_start|>system\\n\' + messages[0][\'content\'] + \'<|im_end|>\\n\' }}\n    {%- else %}\n       